# Part 7 — Advanced Topics

*Last updated:* 2026-11-12

This notebook collects advanced, production-oriented topics for power users:

- custom external database profiles (strict vs broad)
- programmatic YAML management (useful in pipelines)
- troubleshooting & diagnostics (common failure modes)
- integration patterns (Snakemake/Nextflow-friendly workflows)

> **Tip:** You don’t need this notebook for day-to-day conversions. It’s here for when you want reproducibility at scale.

## 7.1 — Custom External Database Inclusion

The external YAML is an explicit **contract**: it defines which external namespaces are allowed to influence your graph.

Two common profiles:

- **Strict profile:** a small allowlist, lower ambiguity, smaller downloads.
- **Broad profile:** a larger allowlist (the shipped default is one), more coverage, but more ambiguity.

The YAML of a local repository has a fixed name, `<organism>_externals_modified.yml` (without it, the shipped default is used). A graph
file records the selection of external databases it was built from: loading it under another selection raises
`idtrack.errors.GraphMismatchError` instead of answering from a graph of the other profile. So keep **one local repository per profile**: each
holds its YAML, its tables and its graph.

The next cell writes two profiles with `idtrack.build.default_externals.apply_allowlist`, which switches on exactly the
`{form: {database: [assembly keys]}}` entries of an allowlist in a template (the shipped templates list every database Ensembl has
for the organism, per form: gene, transcript, translation) and reports allowlisted names the template does not have. It writes
nothing over an existing YAML of your local repository.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import yaml

import idtrack
from idtrack.build.default_externals import DEFAULT_EXTERNALS, apply_allowlist
from idtrack.build.externals import write_externals_yaml

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)
print('Local repository:', LOCAL_REPOSITORY)

organism = 'homo_sapiens'
template_path = Path(idtrack.__file__).parent / 'default_config' / f'{organism}_externals_template.yml'
template = yaml.safe_load(template_path.read_text(encoding='utf-8'))
print('Template:', template_path.name)

# Allowlists name the databases of each form as the template does: {form: {database: [assembly keys]}}.
strict_allowlist = {
    'gene': {'HGNC Symbol': [36, 37, 38], 'EntrezGene': [36, 37, 38]},
    'transcript': {'RefSeq_mRNA': [36, 37, 38]},
    'translation': {'Uniprot/SWISSPROT': [36, 37, 38]},
}
broad_allowlist = DEFAULT_EXTERNALS[organism]  # the selection of the shipped default YAML

for name, allowlist in [('strict', strict_allowlist), ('broad', broad_allowlist)]:
    profile_repository = LOCAL_REPOSITORY.parent / f'idtrack_profile_{name}'  # one local repository per profile
    profile_repository.mkdir(parents=True, exist_ok=True)
    profile, enabled, absent = apply_allowlist(template, organism, allowlist)
    out_path = profile_repository / f'{organism}_externals_modified.yml'
    write_externals_yaml(profile, str(out_path))
    print(f'{name}: {enabled} form/database/assembly entries enabled; not in the template: {absent}')
    print('   written:', out_path)

print()
print('To use a profile: idtrack.API.build(organism, snapshot_release, <its folder>).')

Local repository: idtrack_cache
Template: homo_sapiens_externals_template.yml


strict: 10 form/database/assembly entries enabled; not in the template: []
   written: idtrack_profile_strict/homo_sapiens_externals_modified.yml


broad: 63 form/database/assembly entries enabled; not in the template: []
   written: idtrack_profile_broad/homo_sapiens_externals_modified.yml

To use a profile: idtrack.API.build(organism, snapshot_release, <its folder>).


## 7.2 — Programmatic YAML Management

For pipelines you often want YAML changes to be **scriptable** and **repeatable**.

A safe automation pattern:

1. Start from a template YAML (shipped in `idtrack/default_config/`, or generated as in Part 2).
2. Apply a curated allowlist in code (`apply_allowlist`).
3. Write the resulting YAML to a file you commit alongside your pipeline, and copy it into the local repository of the run.

The next cell shows a reusable helper that:
- reads a template YAML
- applies an allowlist
- writes a modified YAML to the path you give (never over a file you did not name)

> **Tip:** Keep allowlists per organism in your pipeline repository (as Python dicts or a small TOML/YAML). That makes your
> configuration reviewable. `idtrack.build.default_externals.DEFAULT_EXTERNALS` holds the allowlists of the shipped files.

In [2]:
from __future__ import annotations

import os
from pathlib import Path

import yaml

import idtrack
from idtrack.build.default_externals import apply_allowlist
from idtrack.build.externals import write_externals_yaml


def apply_allowlist_to_yaml(template_path: Path, organism: str, allowlist: dict, out_path: Path) -> None:
    template = yaml.safe_load(template_path.read_text(encoding='utf-8'))
    modified, enabled, absent = apply_allowlist(template, organism, allowlist)
    if absent:
        raise ValueError(f'Allowlisted databases that the template does not have: {absent}')
    write_externals_yaml(modified, str(out_path))
    print('Wrote:', out_path)
    print('Enabled form/database/assembly entries:', enabled)


# Example: a small mouse selection, written to a pipeline file (not into a local repository)
organism = 'mus_musculus'
template = Path(idtrack.__file__).parent / 'default_config' / f'{organism}_externals_template.yml'
out_yaml = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache')).parent / f'{organism}_externals_pipeline.yml'

allowlist_mouse = {
    'gene': {'MGI Symbol': [37, 38, 39], 'EntrezGene': [37, 38, 39]},
    'transcript': {'RefSeq_mRNA': [37, 38, 39]},
    'translation': {'Uniprot/SWISSPROT': [37, 38, 39]},
}
apply_allowlist_to_yaml(template, organism=organism, allowlist=allowlist_mouse, out_path=out_yaml)

Wrote: mus_musculus_externals_pipeline.yml
Enabled form/database/assembly entries: 12


## 7.3 — Troubleshooting & Diagnostics

Common failure modes (and what they usually mean):

- **Permission errors in `IDTRACK_LOCAL_REPO`:** your cache directory is not writable.
- **REST timeouts:** network issues or Ensembl REST is temporarily slow.
- **MySQL connection errors:** outbound MySQL ports may be blocked; IDTrack probes a port once and, when it fails, uses the HTTPS/FTP
  dumps for the next ten minutes or longer (the same tables, rows and values, slower per table). Set `IDTRACK_ENSEMBL_ACCESS=https`
  (or `idtrack.settings.configure_ensembl(access='https')`) to skip MySQL entirely. Port `3337` is Ensembl's archive server (GRCh37 from release 79; IDTrack reads it for human GRCh37 only; its
  frozen copies of other assemblies carry later release numbers and are not used).
- **`idtrack.errors.GraphMismatchError` when loading a graph:** the graph file in your local repository was built from another selection of
  external databases, another organism table or by an older graph format than you ask for now; the message names the differences.
  Use another local repository, or build again with `API.build(..., rebuild=True)`.
- **`idtrack.errors.EnsemblDownloadError` / `idtrack.errors.EnsemblRestError`:** Ensembl did not answer after the retries; the message names the URL.
  A release is never silently skipped because a request failed.
- **A warning that releases have no entry in the external YAML:** your external YAML does not include every release of the snapshot
  window; a graph built so has no external identifiers at those releases. Extend the file with `ExternalDatabases.extend_modified_yaml`
  (Part 2, section 2.6).
- **Unexpected 1→n explosion:** your external allowlist is too broad or contains promiscuous namespaces.

The next cell is a compact diagnostic report you can paste into issues or lab notes.

In [ ]:
from __future__ import annotations

import os
import socket
from pathlib import Path

report = {}

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

report['local_repository'] = str(LOCAL_REPOSITORY)
report['local_repository_writable'] = os.access(LOCAL_REPOSITORY, os.W_OK)

# YAML + graph snapshot inventory
report['yaml_files'] = sorted(p.name for p in LOCAL_REPOSITORY.glob('*_externals_modified.yml'))
report['graph_snapshots'] = sorted(p.name for p in LOCAL_REPOSITORY.glob('graph_*.npz'))

# REST connectivity
try:
    import requests

    try:
        r = requests.get('https://rest.ensembl.org/info/ping', headers={'Content-Type': 'application/json'}, timeout=15)
        report['ensembl_rest_status'] = r.status_code
    except Exception as e:
        report['ensembl_rest_status'] = f'failed: {e.__class__.__name__}'
except Exception as e:
    report['ensembl_rest_status'] = f'requests_missing: {e.__class__.__name__}'

# MySQL ports (best-effort)
# Note: port 3337 is Ensembl's archive server; IDTrack reads it for human GRCh37 only. Other releases use 3306/5306.
# IDTrack itself probes a port once and remembers the answer (see `idtrack.build.ensembl_access`).
try:
    import idtrack.settings

    host = idtrack.settings.ENSEMBL.mysql_host
    port_status = {}
    for port in [3306, 5306, 3337]:
        try:
            with socket.create_connection((host, port), timeout=2):
                port_status[port] = 'ok'
        except OSError as e:
            port_status[port] = e.__class__.__name__
    report['ensembl_mysql_ports'] = port_status
except Exception as e:
    report['ensembl_mysql_ports'] = f'skipped: {e.__class__.__name__}'

# Print report
for k, v in report.items():
    print(f'{k}: {v}')

# Optional: quick integrity check (only if the human graph of snapshot 115 exists)
# NOTE: the self-tests (GraphChecks) read Ensembl's tables too (a database manager); we only run one check of the graph here.
if any(g.startswith('graph_homo_sapiens_assembly38_min48_max115') for g in report['graph_snapshots']):
    try:
        from idtrack import API
        from idtrack.checks import GraphChecks

        API.configure_logger()
        api = API.build('homo_sapiens', 115, str(LOCAL_REPOSITORY))  # loads the file of the pinned snapshot
        manager = API.database_manager('homo_sapiens', 115, str(LOCAL_REPOSITORY))
        check = GraphChecks(api.graph, manager).is_edge_with_same_nts_only_at_backbone_nodes()  # `holds`, `details`
        print()
        print('Quick self-test (edge nts invariant):', check.holds)
    except Exception as e:
        print()
        print('Self-test quick check skipped/failed ->', repr(e))

## 7.4 — Integration Patterns

IDTrack works best in pipelines when you make the snapshot boundary and cache location explicit.

Key ideas:

- Set `IDTRACK_LOCAL_REPO` to a stable, shared path (per project or per compute environment).
- Build snapshots once, then reuse them across jobs (a graph file loads without network and is only read, so parallel jobs can share
  one local repository).
- Pin the snapshot release in the pipeline: the graph file name, and the answers, depend on it.
- Store your external YAML alongside your pipeline so the configuration is reviewable.

Below are lightweight patterns you can adapt.


In [4]:
# Snakemake / Nextflow snippets (printed as plain text)

snakemake_rule = """
rule idtrack_build_human_graph:
    output:
        'idtrack_cache/graph_homo_sapiens_assembly38_min48_max115.npz'
    shell:
        "python - <<'PY'\nimport os\nfrom pathlib import Path\nimport idtrack\n\nrepo = os.path.abspath('idtrack_cache')\nPath(repo).mkdir(parents=True, exist_ok=True)\n\nidtrack.API.build('homo_sapiens', 115, repo)\nprint('Built: homo_sapiens 115')\nPY"
""".strip()

nextflow_process = """
process BUILD_IDTRACK_HUMAN {
  output:
    path "idtrack_cache/graph_homo_sapiens_assembly38_min48_max115.npz"
  script:
    "export IDTRACK_LOCAL_REPO=$PWD/idtrack_cache\npython - <<'PY'\nimport os\nfrom pathlib import Path\nimport idtrack\n\nrepo = os.environ['IDTRACK_LOCAL_REPO']\nPath(repo).mkdir(parents=True, exist_ok=True)\n\nidtrack.API.build('homo_sapiens', 115, repo)\nprint('Built: homo_sapiens 115')\nPY"
}
""".strip()

print('--- Snakemake example ---')
print(snakemake_rule)
print('--- Nextflow example ---')
print(nextflow_process)

--- Snakemake example ---
rule idtrack_build_human_graph:
    output:
        'idtrack_cache/graph_homo_sapiens_assembly38_min48_max115.npz'
    shell:
        "python - <<'PY'
import os
from pathlib import Path
import idtrack

repo = os.path.abspath('idtrack_cache')
Path(repo).mkdir(parents=True, exist_ok=True)

idtrack.API.build('homo_sapiens', 115, repo)
print('Built: homo_sapiens 115')
PY"
--- Nextflow example ---
process BUILD_IDTRACK_HUMAN {
  output:
    path "idtrack_cache/graph_homo_sapiens_assembly38_min48_max115.npz"
  script:
    "export IDTRACK_LOCAL_REPO=$PWD/idtrack_cache
python - <<'PY'
import os
from pathlib import Path
import idtrack

repo = os.environ['IDTRACK_LOCAL_REPO']
Path(repo).mkdir(parents=True, exist_ok=True)

idtrack.API.build('homo_sapiens', 115, repo)
print('Built: homo_sapiens 115')
PY"
}
